In [174]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor


In [175]:

folder = r"C:\Users\PC\Desktop\Datachallenge\Dataset"   
datasets = {}  

for file in os.listdir(folder):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder, file))
        datasets[file] = df  

C:\Users\PC\AppData\Local\Temp\ipykernel_9600\1089004864.py:6: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(folder, file))
C:\Users\PC\AppData\Local\Temp\ipykernel_9600\1089004864.py:6: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(folder, file))


In [176]:






X_train = datasets["train_input.csv"]
y_train = datasets["train_outputY.csv"]
X_test = datasets["test_input.csv"]



Traitement

In [177]:
print("=== PRÉPARATION ET IMPUTATION DES DONNÉES ===")

import numpy as np
from sklearn.preprocessing import LabelEncoder

NA_THRESHOLD = 0.90  # seuil de suppression des colonnes trop manquantes

# =========================
# 1. Nettoyage initial
# =========================
X_train_clean = X_train.drop(columns=["ID"]).copy()
X_test_clean  = X_test.drop(columns=["ID"]).copy()

# =========================
# 2. Extraction des cibles
# =========================
y_freq   = y_train["FREQ"].copy()
y_cm     = y_train["CM"].copy()
y_charge = y_train["CHARGE"].copy()
exposure = y_train["ANNEE_ASSURANCE"].copy()

# =========================
# 3. Features uniquement
# =========================
X_train_feat = X_train_clean.drop(columns=["ANNEE_ASSURANCE"]).copy()
X_test_feat  = X_test_clean.drop(columns=["ANNEE_ASSURANCE"]).copy()

# =========================
# 4. Suppression colonnes trop NA
# =========================
cols_to_drop = X_train_feat.columns[
    X_train_feat.isnull().mean() > NA_THRESHOLD
].tolist()

X_train_feat.drop(columns=cols_to_drop, inplace=True, errors="ignore")
X_test_feat.drop(columns=cols_to_drop, inplace=True, errors="ignore")

print(f"Colonnes supprimées (> {NA_THRESHOLD*100:.0f}% NA) : {len(cols_to_drop)}")

# =========================
# 5. Typage
# =========================
num_cols = X_train_feat.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train_feat.select_dtypes(include="object").columns.tolist()

encoders = {}

# =========================
# 6. Imputation + Encodage
# =========================
for col in X_train_feat.columns:

    # Flag NA
    if X_train_feat[col].isnull().any():
        X_train_feat[f"FLAG_NA_{col}"] = X_train_feat[col].isnull().astype(float)
        X_test_feat[f"FLAG_NA_{col}"]  = X_test_feat[col].isnull().astype(float)

    if col in num_cols:
        median_val = X_train_feat[col].median()
        X_train_feat[col].fillna(median_val, inplace=True)
        X_test_feat[col].fillna(median_val, inplace=True)

    elif col in cat_cols:
        X_train_feat[col] = X_train_feat[col].fillna("Missing").astype(str)
        X_test_feat[col]  = X_test_feat[col].fillna("Missing").astype(str)

        le = LabelEncoder()
        X_train_feat[col] = le.fit_transform(X_train_feat[col])

        mapping = dict(zip(le.classes_, range(len(le.classes_))))
        X_test_feat[col] = X_test_feat[col].map(mapping).fillna(0).astype(int)

        encoders[col] = le

# =========================
# 7. Alignement final
# =========================
final_cols = X_train_feat.columns.tolist()

for col in final_cols:
    if col not in X_test_feat.columns:
        X_test_feat[col] = 0.0

X_test_feat = X_test_feat[final_cols].copy()

# =========================
# 8. Matrices finales
# =========================
X_train_model = X_train_feat.astype(float)
X_test_model  = X_test_feat.astype(float)

print("X_train_model shape :", X_train_model.shape)
print("X_test_model shape  :", X_test_model.shape)


=== PRÉPARATION ET IMPUTATION DES DONNÉES ===
Colonnes supprimées (> 90% NA) : 7


C:\Users\PC\AppData\Local\Temp\ipykernel_9600\4023643668.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train_feat[col].fillna(median_val, inplace=True)
C:\Users\PC\AppData\Local\Temp\ipykernel_9600\4023643668.py:61: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

X_train_model shape : (383610, 636)
X_test_model shape  : (95852, 636)


In [178]:
X_train.head
print("Traitement des valeurs manquantes dans les colonnes numériques...")

numeric_columns = X_train.select_dtypes(include=['number']).columns

# Remplissage des colonnes numériques
X_train[numeric_columns] = X_train[numeric_columns].fillna(0)
X_test[numeric_columns] = X_test[numeric_columns].fillna(0)

# Colonnes non numériques
fill_cols = [col for col in X_train.columns if col not in numeric_columns]

# Remplissage des colonnes non numériques
X_train[fill_cols] = X_train[fill_cols].fillna(-999)
X_test[fill_cols] = X_test[fill_cols].fillna(-999)


Traitement des valeurs manquantes dans les colonnes numériques...


Suppression colonne

In [179]:
print("Suppression des colonnes inutiles...")

annee_assurance = X_train["ANNEE_ASSURANCE"]  # utile pour CHARGE
annee_assurance_test = X_test["ANNEE_ASSURANCE"]

X_train = X_train.drop(["ID", "ANNEE_ASSURANCE"], axis=1)
X_test = X_test.drop(["ID", "ANNEE_ASSURANCE"], axis=1)

print("Suppression terminée.")


Suppression des colonnes inutiles...
Suppression terminée.


Encodage

In [180]:
from category_encoders import CountEncoder

print("Encodage CountEncoder...")

encoder = CountEncoder(cols=fill_cols)
encoder.fit(X_train)

X_train_enc = encoder.transform(X_train)
X_test_enc = encoder.transform(X_test)




Encodage CountEncoder...


In [181]:
freq = y_train["FREQ"]
cm = y_train["CM"]


In [182]:
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# Standardisation indispensable
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_enc),
    columns=X_train_enc.columns
)

# Ajout de constante
X_freq = sm.add_constant(X_train_scaled)

# GLM Poisson régularisé
poisson_model = sm.GLM(freq, X_freq, family=sm.families.Poisson())

print("Entraînement du GLM Poisson régularisé...")
poisson_results = poisson_model.fit_regularized(
    method="l1",
    alpha=1.0,
    L1_wt=0.0  # 100% Ridge
)
print("Terminé.")

freq_pred = poisson_results.predict(X_freq)


Entraînement du GLM Poisson régularisé...
Terminé.


CM

In [183]:
print("Préparation des données pour le modèle CM...")

train_cm = pd.concat([X_train_enc, y_train], axis=1)
train_cm = train_cm[train_cm["FREQ"] > 0]  # uniquement sinistres positifs

X_cm_raw = train_cm[X_train_enc.columns]
cm = train_cm["CM"]


Préparation des données pour le modèle CM...


In [184]:
print("Standardisation des données pour le modèle CM...")

scaler_cm = StandardScaler()

X_cm_scaled = pd.DataFrame(
    scaler_cm.fit_transform(X_cm_raw),
    columns=X_cm_raw.columns
)

# Ajout d'une constante pour GLM
X_cm_scaled_const = sm.add_constant(X_cm_scaled)


Standardisation des données pour le modèle CM...


In [185]:
print("Préparation du modèle CM...")

# 1. Sélection uniquement FREQ > 0
train_cm = pd.concat([X_train_enc, y_train], axis=1)
train_cm = train_cm[train_cm["FREQ"] > 0]

X_cm_raw = train_cm[X_train_enc.columns]
cm = train_cm["CM"]

# 2. Standardisation avec index réaligné
scaler_cm = StandardScaler()

X_cm_scaled = pd.DataFrame(
    scaler_cm.fit_transform(X_cm_raw),
    columns=X_cm_raw.columns,
    index=X_cm_raw.index  
)

# 3. Ajout de constante
X_cm_scaled_const = sm.add_constant(X_cm_scaled)

# 4. GLM Tweedie régularisé
print("Entraînement du modèle CM (GLM Tweedie régularisé)...")

tweedie_model = sm.GLM(
    cm,
    X_cm_scaled_const,
    family=sm.families.Tweedie(var_power=1.5, link=sm.families.links.log())
)

tweedie_results = tweedie_model.fit_regularized(
    method="l1",
    alpha=1.0,
    L1_wt=0.0
)

print("Modèle CM entraîné.")

# 5. Prédiction sur tout le jeu d'entraînement
X_cm_all_scaled = pd.DataFrame(
    scaler_cm.transform(X_train_enc),
    columns=X_train_enc.columns
)

X_cm_all_scaled_const = sm.add_constant(X_cm_all_scaled)

cm_pred = tweedie_results.predict(X_cm_all_scaled_const)
print("Prédictions CM terminées.")


Préparation du modèle CM...
Entraînement du modèle CM (GLM Tweedie régularisé)...


c:\Users\PC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


Modèle CM entraîné.
Prédictions CM terminées.


In [186]:
from sklearn.metrics import mean_squared_error

charge_pred = freq_pred * cm_pred * annee_assurance


# Calcul MSE puis RMSE manuellement
mse_glm = mean_squared_error(y_train["CHARGE"], charge_pred)
rmse_glm = mse_glm ** 0.5

print("RMSE final GLM :", rmse_glm)


RMSE final GLM : 6803.015137711734


## Modèles de machine learning

In [187]:
from sklearn.model_selection import train_test_split

y_charge = y_train["CHARGE"]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_enc, y_charge, test_size=0.2, random_state=42
)


Modèle XGBoost

In [188]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    tree_method="hist"  # rapide
)

print("Entraînement du modèle XGBoost...")
xgb_model.fit(X_tr, y_tr)
print("Modèle entraîné.")


Entraînement du modèle XGBoost...
Modèle entraîné.


In [189]:
from sklearn.metrics import mean_squared_error
import numpy as np

y_val_pred = xgb_model.predict(X_val)

rmse_xgb = np.sqrt(mean_squared_error(y_val, y_val_pred))
print("RMSE XGBoost :", rmse_xgb)


RMSE XGBoost : 7288.956173426428


In [190]:
test_pred = xgb_model.predict(X_test_enc)


In [191]:
X_test_raw = pd.read_csv("test_input.csv")

submission = pd.DataFrame({
    "ID": X_test_raw["ID"],
    "CHARGE": test_pred
})

submission.to_csv("submission_xgb.csv", index=False)
print("Fichier de soumission généré : submission_xgb.csv")


C:\Users\PC\AppData\Local\Temp\ipykernel_9600\3726943740.py:1: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  X_test_raw = pd.read_csv("test_input.csv")


Fichier de soumission généré : submission_xgb.csv


In [192]:
y_train_cm = y_train["CM"].clip(lower=0)


In [193]:
from sklearn.model_selection import train_test_split

# Split cohérent sur X, y et ANNEE_ASSURANCE
X_tr, X_val, y_tr, y_val, annee_tr, annee_val = train_test_split(
    X_train_enc,
    y_train,           
    annee_assurance,
    test_size=0.2,
    random_state=42
)

freq_tr = y_tr["FREQ"]
freq_val = y_val["FREQ"]

cm_tr = y_tr["CM"]
cm_val = y_val["CM"]

charge_val_true = y_val["CHARGE"]


In [194]:
import numpy as np

cm_tr  = np.maximum(cm_tr, 1e-6)
cm_val = np.maximum(cm_val, 1e-6)
cm     = np.maximum(cm, 1e-6)


Entraînement modèle  LightGBM

In [195]:
import lightgbm as lgb
freq_model = lgb.LGBMRegressor(
    objective="poisson",
    metric="rmse",
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42
)

print("Entraînement LightGBM sur la fréquence...")
freq_model.fit(
    X_tr, freq_tr,
    eval_set=[(X_val, freq_val)],
    eval_metric="rmse"
)

freq_val_pred = freq_model.predict(X_val)


Entraînement LightGBM sur la fréquence...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.236361 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3142
[LightGBM] [Info] Number of data points in the train set: 306888, number of used features: 370
[LightGBM] [Info] Start training from score -4.417608


In [196]:
import numpy as np

cm_tr  = np.maximum(cm_tr, 1e-6)
cm_val = np.maximum(cm_val, 1e-6)
cm     = np.maximum(cm, 1e-6)


In [197]:
cm_model = lgb.LGBMRegressor(
    objective="Tweedie",   # 🔁 au lieu de "tweedie"
    metric="rmse",
    n_estimators=1500,
    learning_rate=0.01,
    num_leaves=63,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42
)

print("Entraînement LightGBM sur le coût moyen...")
cm_model.fit(
    X_tr, cm_tr,
    eval_set=[(X_val, cm_val)],
    eval_metric="rmse"
)

cm_val_pred = cm_model.predict(X_val)


Entraînement LightGBM sur le coût moyen...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.220728 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3142
[LightGBM] [Info] Number of data points in the train set: 306888, number of used features: 370
[LightGBM] [Info] Start training from score 5.185523


In [198]:


from sklearn.metrics import mean_squared_error
import numpy as np

# Reconstitution de la charge
charge_val_pred = freq_val_pred * cm_val_pred * annee_val

# Calcul du RMSE validation
rmse_val = np.sqrt(mean_squared_error(y_val["CHARGE"], charge_val_pred))

print("RMSE validation LightGBM (FREQ × CM × ANNEE_ASSURANCE) :", rmse_val)


RMSE validation LightGBM (FREQ × CM × ANNEE_ASSURANCE) : 7046.466676802109


In [199]:
freq_model_full = lgb.LGBMRegressor(
    objective="poisson",
    metric="rmse",
    n_estimators=1000,
    learning_rate=0.01,
    num_leaves=123,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

freq_model_full.fit(X_train_enc, y_train["FREQ"])


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.290065 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3151
[LightGBM] [Info] Number of data points in the train set: 383610, number of used features: 370
[LightGBM] [Info] Start training from score -4.385889


,boosting_type,'gbdt'
,num_leaves,123
,max_depth,-1
,learning_rate,0.01
,n_estimators,1000
,subsample_for_bin,200000
,objective,'poisson'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [200]:
import numpy as np

cm_tr  = np.maximum(cm_tr, 1e-6)
cm_val = np.maximum(cm_val, 1e-6)
cm     = np.maximum(cm, 1e-6)


In [201]:
y_train_cm = y_train["CM"].clip(lower=1e-6)
y_train_cm = np.nan_to_num(y_train_cm, nan=1e-6, neginf=1e-6)

In [202]:
cm_model_full = lgb.LGBMRegressor(
    objective="Tweedie",
    metric="rmse",
    n_estimators=1000,
    learning_rate=0.01,
    num_leaves=123,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

cm_model_full.fit(X_train_enc, y_train["CM"])


LightGBMError: [tweedie]: at least one target label is negative

In [ ]:
print("Prédictions sur le test...")

freq_test_pred = freq_model_full.predict(X_test_enc)
cm_test_pred   = cm_model_full.predict(X_test_enc)

charge_test_pred = freq_test_pred * cm_test_pred * annee_assurance_test


Prédictions sur le test...


In [ ]:
submission = pd.DataFrame({
    "ID": X_test_raw["ID"],    
    "CHARGE": charge_test_pred
})

submission.to_csv("submission_ligbm.csv", index=False)

print("Fichier de soumission créé : sub_ligbm.csv")


Fichier de soumission créé : submission_ligbm.csv


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np


In [ ]:
cm = y_train["CM"].values
freq = y_train["FREQ"].values
annee_assurance = y_train["ANNEE_ASSURANCE"].values


In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, freq_tr, freq_val = train_test_split(
    X_train_enc, freq, test_size=0.2, random_state=42
)

_, _, cm_tr, cm_val = train_test_split(
    X_train_enc, cm, test_size=0.2, random_state=42
)

_, _, annee_tr, annee_val = train_test_split(
    X_train_enc, annee_assurance, test_size=0.2, random_state=42
)


In [ ]:
freq_model = HistGradientBoostingRegressor(
    loss="poisson",
    max_depth=None,
    max_iter=500,
    learning_rate=0.05,
    l2_regularization=0.0,
    max_bins=255,
    random_state=42
)

print("Entraînement HGB sur la fréquence...")
freq_model.fit(X_tr, freq_tr)

freq_val_pred = freq_model.predict(X_val)


Entraînement HGB sur la fréquence...


In [ ]:
cm_tr = cm_tr.clip(min=1e-6)
cm_val = cm_val.clip(min=1e-6)


In [ ]:
cm_model = HistGradientBoostingRegressor(
    loss="gamma",     # parfait pour coût moyen > 0
    max_depth=None,
    max_iter=700,
    learning_rate=0.03,
    l2_regularization=0.0,
    max_bins=255,
    random_state=42
)

print("Entraînement HGB sur le coût moyen...")
cm_model.fit(X_tr, cm_tr)

cm_val_pred = cm_model.predict(X_val)


Entraînement HGB sur le coût moyen...


In [ ]:
charge_val_pred = freq_val_pred * cm_val_pred * annee_val

rmse_val = np.sqrt(mean_squared_error(y_val["CHARGE"], charge_val_pred))

print("RMSE validation HGB :", rmse_val)


RMSE validation HGB : 7045.217724999518


In [ ]:
# Correction CM global pour le modèle final
epsilon = 1e-6
cm = cm.clip(min=epsilon)

print("Entraînement final HGB (CM) sur tout le train...")
cm_model_full = HistGradientBoostingRegressor(
    loss="gamma",
    max_depth=None,
    max_iter=700,
    learning_rate=0.03,
    max_bins=255,
    random_state=42
)

cm_model_full.fit(X_train_enc, cm)


Entraînement final HGB (CM) sur tout le train...


,loss,'gamma'
,quantile,None
,learning_rate,0.03
,max_iter,700
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'


In [ ]:
freq_test_pred = freq_model_full.predict(X_test_enc)
cm_test_pred   = cm_model_full.predict(X_test_enc)

charge_test_pred = freq_test_pred * cm_test_pred * annee_assurance_test


In [ ]:
submission = pd.DataFrame({
    "ID": X_test_raw["ID"],
    "CHARGE": charge_test_pred
})

submission.to_csv("submission_hgb.csv", index=False)

print("Fichier généré : submission_hgb.csv")


Fichier généré : submission_hgb.csv


Modèle non vu en cours: TabNet